# 03 · Pipeline: de FS a ML (objetivo y partición temporal)

**ML** = variables de FS + **objetivo** (`y`) + **partición**.

- `y = 1` si el cliente compra al menos una vez en los **90 días siguientes** al corte; `0` si no.
- La partición es **temporal**: se entrena con el pasado y se evalúa con un periodo que el modelo no vio. **Nunca aleatoria.**

Necesita FS construida (cuaderno 02).

In [ ]:
# Parámetros. None usa las rutas por defecto del proyecto (o las variables de entorno CURSO_DB y CURSO_REPORTES).
RUTA_DB = None
RUTA_REPORTES = None
VENTANA_OBJETIVO_DIAS = 90
CORTES = {
    'train': ['2010-06-01', '2010-07-01', '2010-08-01', '2010-09-01', '2010-10-01', '2010-11-01', '2010-12-01'],
    'valid': ['2011-03-01'],
    'test': ['2011-06-01', '2011-09-01'],
}

In [ ]:
import os, sqlite3, sys
from pathlib import Path

import pandas as pd

pd.set_option('display.width', 150, 'display.max_columns', 30)

# Carpeta del proyecto: la que contiene «sql» y «pyproject.toml».
RAIZ = Path.cwd().resolve()
while not ((RAIZ / 'sql').exists() and (RAIZ / 'pyproject.toml').exists()) and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
SQL = RAIZ / 'sql'
DB = Path(RUTA_DB or os.environ.get('CURSO_DB') or RAIZ / 'data' / 'processed' / 'curso.db')
DB.parent.mkdir(parents=True, exist_ok=True)

con = sqlite3.connect(DB)
con.execute('PRAGMA foreign_keys = ON')


def ejecutar_script(archivo):
    """Ejecuta un archivo .sql de la carpeta sql/ (puede tener varias sentencias)."""
    con.executescript((SQL / archivo).read_text(encoding='utf-8'))
    con.commit()


def escalar(consulta, parametros=None):
    fila = con.execute(consulta, parametros or {}).fetchone()
    return None if fila is None else fila[0]


print('Proyecto:', RAIZ)
print('Base de datos:', DB)
REPORTES = Path(RUTA_REPORTES or os.environ.get('CURSO_REPORTES') or RAIZ / 'reports')
(REPORTES / 'figures').mkdir(parents=True, exist_ok=True)

## 1. Crear las tablas y registrar las particiones
Antes de continuar se comprueba que la **ventana del objetivo cabe en los datos**: si se saliera del último día con datos, el objetivo estaría incompleto y sesgado.

In [ ]:
ejecutar_script('ml/01_crear_ml.sql')
ultimo_dia = escalar('SELECT MAX(fecha_factura) FROM master_facturas')[:10]
print('Último día con datos:', ultimo_dia)
for particion, cortes in CORTES.items():
    for corte in cortes:
        fin = escalar("SELECT date(:c, '+' || :d || ' day')", {'c': corte, 'd': VENTANA_OBJETIVO_DIAS})
        assert fin <= ultimo_dia, f'El corte {corte} necesita datos hasta {fin}, pero el último día con datos es {ultimo_dia}.'
        con.execute('INSERT INTO ml_particiones (fecha_corte, particion, ventana_dias, fin_ventana) VALUES (?,?,?,?)',
                    (corte, particion, VENTANA_OBJETIVO_DIAS, fin))
con.commit()
pd.read_sql('SELECT * FROM ml_particiones ORDER BY fecha_corte', con)

## 2. Poblar ML
`sql/ml/02_poblar_ml.sql` une las variables de FS con el objetivo y la partición. El objetivo mira el **futuro** del corte (esa es su función); las variables solo miran el **pasado**.

In [ ]:
ejecutar_script('ml/02_poblar_ml.sql')
pd.read_sql("""SELECT particion, COUNT(*) AS filas, SUM(y) AS recompran, ROUND(100.0 * AVG(y), 1) AS tasa_recompra_pct
               FROM ml_dataset GROUP BY particion ORDER BY MIN(fecha_corte)""", con)

## 3. Ventanas del objetivo de cada corte
Las ventanas **no se solapan** entre particiones.

In [ ]:
import matplotlib
matplotlib.use('Agg') if os.environ.get('CURSO_SIN_PANTALLA') else None
import matplotlib.pyplot as plt

p = pd.read_sql('SELECT * FROM ml_particiones ORDER BY fecha_corte', con)
colores = {'train': '#2bb3ff', 'valid': '#fcd34d', 'test': '#34d399'}
fig, ax = plt.subplots(figsize=(11, 3))
for i, r in p.iterrows():
    ini, fin = pd.Timestamp(r.fecha_corte), pd.Timestamp(r.fin_ventana)
    ax.barh(i, (fin - ini).days, left=ini, color=colores[r.particion], edgecolor='k')
ax.set_yticks(range(len(p)))
ax.set_yticklabels([f'{r.fecha_corte} ({r.particion})' for r in p.itertuples()])
ax.set_title('Ventana del objetivo de cada corte (corte → corte + 90 días)')
plt.tight_layout()
fig.savefig(REPORTES / 'figures' / 'ml_ventanas.png', dpi=110)
plt.show()

## 4. Comprobar el objetivo con pandas
Recalculamos `y` para una muestra de filas, mirando las facturas **dentro de la ventana**, y comparamos con lo que calculó SQL.

In [ ]:
ml = pd.read_sql('SELECT cliente_id, fecha_corte, y FROM ml_dataset', con).sample(min(200, escalar('SELECT COUNT(*) FROM ml_dataset')), random_state=0)
ventas = pd.read_sql("""SELECT f.cliente_id, f.fecha_factura FROM master_lineas l JOIN master_facturas f ON f.factura_id = l.factura_id
                        WHERE l.es_venta_valida = 1 AND f.cliente_id IS NOT NULL""", con)
ventas['fecha'] = pd.to_datetime(ventas['fecha_factura'])


def y_pandas(fila):
    ini = pd.Timestamp(fila.fecha_corte)
    fin = ini + pd.Timedelta(days=VENTANA_OBJETIVO_DIAS)
    return int(((ventas['cliente_id'] == fila.cliente_id) & (ventas['fecha'] >= ini) & (ventas['fecha'] < fin)).any())


coincidencia = (ml.apply(y_pandas, axis=1) == ml['y']).mean()
print(f'Coincidencia entre SQL y pandas en {len(ml)} filas: {coincidencia:.0%}')
assert coincidencia == 1.0

In [ ]:
con.close()
print('Listo: ML construida en', DB)

## Para practicar
1. Cambia `VENTANA_OBJETIVO_DIAS` a 60 y vuelve a ejecutar el cuaderno desde el principio. ¿Cómo cambia la tasa de recompra? Observa que cambian también los cortes posibles.
2. ¿Qué pasaría si la partición fuera **aleatoria**? Escribe qué fuga de información introduciría.